In [1]:
import pandas as pd

In [2]:
# 1. Load dataset
df = pd.read_csv("India_ecommerce_orders_dirty_dataset.csv")

print("Original Shape:", df.shape)
print(df.info())
print(df.isnull().sum())

Original Shape: (50350, 19)
<class 'pandas.DataFrame'>
RangeIndex: 50350 entries, 0 to 50349
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       48810 non-null  str    
 1   CustomerName     49379 non-null  str    
 2   Gender           49335 non-null  str    
 3   Age              49340 non-null  float64
 4   City             49370 non-null  str    
 5   Country          50350 non-null  str    
 6   CustomerType     48832 non-null  str    
 7   SignupDate       50350 non-null  str    
 8   OrderID          50350 non-null  str    
 9   OrderDate        50350 non-null  str    
 10  ProductID        50350 non-null  str    
 11  ProductName      50350 non-null  str    
 12  ProductCategory  50350 non-null  str    
 13  Quantity         50350 non-null  int64  
 14  UnitPrice        50350 non-null  str    
 15  Discount         50350 non-null  int64  
 16  Revenue          49326 non-null  float64


In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(df.columns.tolist())

['customerid', 'customername', 'gender', 'age', 'city', 'country', 'customertype', 'signupdate', 'orderid', 'orderdate', 'productid', 'productname', 'productcategory', 'quantity', 'unitprice', 'discount', 'revenue', 'paymentmethod', 'orderstatus']


In [4]:
df = df.drop_duplicates().copy()

print("After duplicate removal:", df.shape)

After duplicate removal: (50100, 19)


In [5]:
text_columns = [
    "customername",
    "gender",
    "city",
    "country",
    "customertype",
    "productname",
    "productcategory",
    "paymentmethod",
    "orderstatus"
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

In [6]:
text_columns = [
    "customername",
    "gender",
    "city",
    "country",
    "customertype",
    "productname",
    "productcategory",
    "paymentmethod",
    "orderstatus"
]

for col in text_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.capitalize()
    )

In [7]:
df["customername"] = df["customername"].str.title()
df["gender"] = df["gender"].str.title()
df["city"] = df["city"].str.title()
df["country"] = df["country"].str.title()
df["customertype"] = df["customertype"].str.title()
df["productname"] = df["productname"].str.title()
df["productcategory"] = df["productcategory"].str.title()
df["paymentmethod"] = df["paymentmethod"].str.title()
df["orderstatus"] = df["orderstatus"].str.title()

In [9]:
df["signupdate"] = pd.to_datetime(
    df["signupdate"],
    errors="coerce"
)

df["orderdate"] = pd.to_datetime(
    df["orderdate"],
    errors="coerce"
)

In [10]:
for col in ["unitprice", "revenue"]:
    df[col] = (
        df[col]
        .astype("string")
        .str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [11]:
for col in ["age", "quantity", "discount"]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [12]:
df.loc[
    (df["age"] < 0) | (df["age"] > 100),
    "age"
] = pd.NA

In [13]:
df["age"] = (
    df.groupby("customerid")["age"]
    .transform(lambda x: x.fillna(x.median()))
)

df["age"] = df["age"].fillna(
    df["age"].median()
)

In [14]:
df = df.dropna(
    subset=["customerid"]
).copy()

In [15]:
for col in ["customername", "gender", "city", "customertype"]:
    
    df[col] = (
        df.groupby("customerid")[col]
        .transform(lambda x: x.ffill().bfill())
    )
    
    df[col] = df[col].fillna("Unknown")

In [16]:
calculated_revenue = (
    df["quantity"]
    * df["unitprice"]
    * (1 - df["discount"] / 100)
)

df["revenue"] = df["revenue"].fillna(
    calculated_revenue
)

In [17]:
rfm_df = df[
    (df["quantity"] > 0) &
    (df["revenue"] > 0) &
    (df["orderdate"].notna())
].copy()

In [18]:
rfm_df = rfm_df[
    ~rfm_df["orderstatus"].isin(
        ["Cancelled", "Refunded"]
    )
].copy()

In [19]:
duplicate_orders = rfm_df[
    rfm_df.duplicated(
        "orderid",
        keep=False
    )
].sort_values("orderid")

print(duplicate_orders)

        customerid    customername  gender   age        city country  \
25542    CUST02978     Rohan Bhatt  Female  51.0       Delhi   India   
48917    CUST02978     Rohan Bhatt  Female  51.0       Delhi   India   
31537    CUST04908     Sai Chauhan  Female  29.0       Surat   India   
35605    CUST04908     Sai Chauhan  Female  29.0       Surat   India   
15760    CUST02926     Ishaan Shah   Other  61.0      Indore   India   
48495    CUST02926     Ishaan Shah   Other  61.0      Indore   India   
10104    cust01290     Rahul Yadav    Male  55.0      Bhopal   India   
13768    cust01290     Rahul Yadav    Male  55.0      Bhopal   India   
47260    cust01290     Rahul Yadav    Male  55.0      Bhopal   India   
21655    CUST00079     Ravi Pillai    Male  50.0       Surat   India   
38924    CUST00079     Ravi Pillai    Male  50.0       Surat   India   
3473     CUST00639       Harsh Das  Female  63.0      Jaipur   India   
23587    CUST00639       Harsh Das  Female  63.0      Jaipur   I

In [20]:
frequency = rfm_df.groupby(
    "customerid"
)["orderid"].nunique()

In [21]:
print("Final Shape:", rfm_df.shape)

print("\nMissing Values:")
print(rfm_df.isnull().sum())

print("\nDuplicate Rows:")
print(rfm_df.duplicated().sum())

print("\nCustomers:")
print(rfm_df["customerid"].nunique())

print("\nOrders:")
print(rfm_df["orderid"].nunique())

print("\nInvalid Quantity:")
print((rfm_df["quantity"] <= 0).sum())

print("\nInvalid Revenue:")
print((rfm_df["revenue"] <= 0).sum())

Final Shape: (8478, 19)

Missing Values:
customerid         0
customername       0
gender             0
age                0
city               0
country            0
customertype       0
signupdate         0
orderid            0
orderdate          0
productid          0
productname        0
productcategory    0
quantity           0
unitprice          0
discount           0
revenue            0
paymentmethod      0
orderstatus        0
dtype: int64

Duplicate Rows:
0

Customers:
3794

Orders:
8461

Invalid Quantity:
0

Invalid Revenue:
0


In [23]:
text_columns = [
    "customername",
    "gender",
    "city",
    "country",
    "customertype",
    "productname",
    "productcategory",
    "paymentmethod",
    "orderstatus"
]

for col in text_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.title()
    )

In [24]:
df[text_columns].head(10)

,customername,gender,city,country,customertype,productname,productcategory,paymentmethod,orderstatus
0,Aditya Bhatt,Male,Ahmedabad,India,New,Puzzle Set,Toys,Net Banking,Delivered
1,Suresh Bhatt,Female,Bengaluru,India,New,Remote Car,Toys,Debit Card,Delivered
2,Simran Bose,Female,Bengaluru,India,New,Self-Help Book,Books,Credit Card,Delivered
3,Navya Bhatt,Male,Pune,India,Premium,Self-Help Book,Books,Upi,Delivered
4,Diya Shah,Female,Nagpur,India,Premium,Atta 10Kg,Groceries,Cash On Delivery,Delivered
5,Arjun Sharma,Female,Pune,India,Returning,Led Bulb,Home & Kitchen,Upi,Returned
6,Myra Patel,Male,Pune,India,Vip,Lipstick,Beauty,Wallet,Returned
7,Aisha Iyer,Male,Hyderabad,India,Premium,Water Bottle,Home & Kitchen,Wallet,Delivered
8,Krishna Joshi,Female,Ahmedabad,India,Returning,Cricket Bat,Sports,Upi,Pending
9,Navya Das,Female,Patna,India,Premium,Laptop,Electronics,Net Banking,Delivered


In [25]:
df.columns = df.columns.str.strip().str.title()

In [28]:
print(df.columns.tolist())

['Customerid', 'Customername', 'Gender', 'Age', 'City', 'Country', 'Customertype', 'Signupdate', 'Orderid', 'Orderdate', 'Productid', 'Productname', 'Productcategory', 'Quantity', 'Unitprice', 'Discount', 'Revenue', 'Paymentmethod', 'Orderstatus']


In [29]:
print("Final Shape:", rfm_df.shape)

print("\nMissing Values:")
print(rfm_df.isnull().sum())

print("\nDuplicate Rows:")
print(rfm_df.duplicated().sum())

print("\nCustomers:")
print(rfm_df["customerid"].nunique())

print("\nOrders:")
print(rfm_df["orderid"].nunique())

print("\nInvalid Quantity:")
print((rfm_df["quantity"] <= 0).sum())

print("\nInvalid Revenue:")
print((rfm_df["revenue"] <= 0).sum())

Final Shape: (8478, 19)

Missing Values:
customerid         0
customername       0
gender             0
age                0
city               0
country            0
customertype       0
signupdate         0
orderid            0
orderdate          0
productid          0
productname        0
productcategory    0
quantity           0
unitprice          0
discount           0
revenue            0
paymentmethod      0
orderstatus        0
dtype: int64

Duplicate Rows:
0

Customers:
3794

Orders:
8461

Invalid Quantity:
0

Invalid Revenue:
0


In [30]:
rfm_df.to_csv(
    "customer_orders_cleaned.csv",
    index=False
)